In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns
df = pd.read_csv(paths.der("macroregion_pop_gdp_traveltime.csv"))

In [ ]:
from palettable.cartocolors.diverging import Geyser_5_r, Geyser_5
from palettable.cartocolors.qualitative import Safe_3, Antique_5, Vivid_7
from palettable.cubehelix import cubehelix3_16
import sys
sys.path.insert(0, '../code')
from constants import CONTINENT_COLORS, REGION_TO_CONTINENT


In [ ]:
df1 = pd.read_csv(paths.der("macroregion_pop_gdp_tt_average.csv"))
df1['continent'] = df1['region'].map(REGION_TO_CONTINENT)
colors = df1['region'].map(REGION_TO_CONTINENT).map(CONTINENT_COLORS)

In [ ]:
df1['gdp_pc'].mean()

In [ ]:
def set_font(ax, size):
    for label in (ax.get_xticklabels() + ax.get_yticklabels()):
        label.set_fontsize(size)
    return ax

In [ ]:
from matplotlib.patches import Patch
import matplotlib.ticker as mticker

reg = pd.read_csv(paths.der("macroregion_pop_gdp_tt_average.csv"))
reg['continent'] = reg['region'].map(REGION_TO_CONTINENT)
colors_reg = reg['continent'].map(CONTINENT_COLORS)

f, axs = plt.subplots(2, 1, figsize=(10, 10))

# ── Panel A: real-value scatter, log GDP scale ──
ax = axs[0]
for _, row in reg.iterrows():
    c = CONTINENT_COLORS[row['continent']]
    ax.scatter(row['gdp_pc'], row['time'],
               s=row['population'] / 2e6, color=c, alpha=0.85,
               edgecolors='white', linewidths=0.5, zorder=3)
    ax.annotate(row['region'], (row['gdp_pc'], row['time']),
                fontsize=7, xytext=(4, 2), textcoords='offset points', color='#333333')

# Quadrant dividers (round, interpretable values)
GDP_DIV = 10_000
TT_DIV  = 30
ax.axvline(GDP_DIV, color='#aaaaaa', ls='--', lw=1, zorder=1)
ax.axhline(TT_DIV,  color='#aaaaaa', ls='--', lw=1, zorder=1)

# Shade double-disadvantage quadrant (low GDP, high TT)
ax.fill_between([1000, GDP_DIV], TT_DIV, 200, color='#f4a582', alpha=0.18, zorder=0)
ax.text(1200, 160, 'Double\ndisadvantage', fontsize=9, color='#c0392b',
        fontstyle='italic', va='top')
ax.text(GDP_DIV * 1.05, 160, 'Low access\nhigh income', fontsize=9, color='#666666',
        fontstyle='italic', va='top')
ax.text(GDP_DIV * 1.05, 5, 'Double\nadvantage', fontsize=9, color='#2980b9',
        fontstyle='italic', va='bottom')
ax.text(1200, 5, 'High access\nlow income', fontsize=9, color='#27ae60',
        fontstyle='italic', va='bottom')

ax.set_xscale('log')
ax.set_xlim([1000, 70_000])
ax.set_ylim([0, 200])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('GDP per capita (PPP 2011 USD)', size=13)
ax.set_ylabel('Mean travel time to forest [min]', size=13)
ax.text(0.01, 0.975, 'A', transform=ax.transAxes, fontsize=20, fontweight='bold', va='top')

continent_patches = [Patch(color=c, label=cont) for cont, c in CONTINENT_COLORS.items()]
ax.legend(handles=continent_patches, bbox_to_anchor=(1.02, 1.01),
          loc='upper left', fontsize=11, title='Continent')

# ── Panel B: GDP per capita of people within 10 min, sorted ──
ax = axs[1]
region_palette = {r: CONTINENT_COLORS[c] for r, c in REGION_TO_CONTINENT.items()}
tmp = df[df['threshold_min'] == 10].reset_index(drop=True)
tmp['continent'] = tmp['region'].map(REGION_TO_CONTINENT)
tmp = tmp.sort_values('gdp_pc')
sns.barplot(x='region', y='gdp_pc', hue='region', ax=ax, data=tmp, palette=region_palette, legend=False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylabel('GDP per capita of people\nwithin 10 min of forest', size=12)
ax.set_xlabel('')
ax.set_xticks(range(len(tmp)))
ax.set_xticklabels(tmp['region'].values, rotation=45, ha='right')
ax.text(0.01, 0.975, 'B', transform=ax.transAxes, fontsize=20, fontweight='bold', va='top')

f.tight_layout()

In [ ]:
import scipy.stats as stats

gdp = reg['gdp_pc'].values
tt_r = reg['time'].values
pop_r = reg['population'].values.astype(float)

# Spearman (n=17)
r_sp, p_sp = stats.spearmanr(gdp, tt_r)

# Population-weighted Pearson
w = pop_r / pop_r.sum()
gm = np.sum(w * gdp);  tm = np.sum(w * tt_r)
cov_w = np.sum(w * (gdp - gm) * (tt_r - tm))
r_wp = cov_w / (np.sqrt(np.sum(w * (gdp - gm)**2)) * np.sqrt(np.sum(w * (tt_r - tm)**2)))

print(f"Spearman r   (n=17, unweighted) : {r_sp:.3f}  p={p_sp:.3f}")
print(f"Pearson r    (pop-weighted)     : {r_wp:.3f}")
print()
print("Double-disadvantage regions (GDP < $10k AND TT > 30 min):")
dd = reg[(reg['gdp_pc'] < 10_000) & (reg['time'] > 30)]
print(dd[['region','time','gdp_pc','population']].to_string(index=False))
print(f"Combined population: {dd['population'].sum()/1e9:.2f} billion")

In [ ]:
fout = paths.fig("Figure3AB.pdf")
f.savefig(fout, dpi=300, bbox_inches='tight')